<a href="https://colab.research.google.com/github/wvb20/cv-face-alignment/blob/main/notebooks/00_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# === Bootstrap — local or Colab ===
import os, subprocess, sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_PATH = '/content/cv-face-alignment'
    if not os.path.exists(REPO_PATH):
        subprocess.run(
            ['git', 'clone', 'https://github.com/wvb20/cv-face-alignment.git', REPO_PATH],
            check=True,
        )
    else:
        subprocess.run(['git', '-C', REPO_PATH, 'pull'], check=True)
else:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'config.py').exists():
            REPO_PATH = str(candidate)
            break
    else:
        raise RuntimeError(
            'Could not find the repo root. Start Jupyter from the repository or set CV_FACE_ALIGNMENT_REPO_ROOT.'
        )

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

from src.runtime import configure_notebook_runtime
runtime = configure_notebook_runtime(REPO_PATH)
from src import config


print('✓ Setup complete')
runtime_label = 'Colab' if runtime.in_colab else 'Local'
print(f'Runtime:   {runtime_label}')
print(f'Repo:      {runtime.repo_root}')
print(f'Workspace: {runtime.workspace_dir}')
print(f'Data dir:  {config.DATA_DIR}')
print(f'Figures:   {config.FIGURES_DIR}')

✓ Setup complete
Runtime:   Local
Repo:      /Users/willvaughn/Developer/cv-face-alignment
Workspace: /Users/willvaughn/Developer/cv-face-alignment
Data dir:  /Users/willvaughn/Developer/cv-face-alignment/data
Figures:   /Users/willvaughn/Developer/cv-face-alignment/figures


In [2]:
# === Confirm runtime environment ===
import sys, torch, cv2, numpy as np, sklearn

print(f'Python:       {sys.version.split()[0]}')
print(f'NumPy:        {np.__version__}')
print(f'OpenCV:       {cv2.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'PyTorch:      {torch.__version__}')
print(f'CUDA avail:   {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:          {torch.cuda.get_device_name(0)}')

Python:       3.11.14
NumPy:        2.4.4
OpenCV:       4.13.0
scikit-learn: 1.8.0
PyTorch:      2.10.0
CUDA avail:   False


In [3]:
# === Cell 3: Confirm repo root and Python path ===
print(f'✓ Repo at: {REPO_PATH}')
print(f'✓ Project root on Python path')
print(f'✓ Python path contains repo: {REPO_PATH in sys.path}')

✓ Repo at: /Users/willvaughn/Developer/cv-face-alignment
✓ Project root on Python path
✓ Python path contains repo: True


In [4]:
# === Cell 4: Verify src/ modules import cleanly ===
from src import config, data, evaluate, io, features, models, visualise
print('✓ All modules import cleanly')
print(f'  Image size: {config.IMAGE_SIZE}')
print(f'  Landmarks per face: {config.N_LANDMARKS}')
print(f'  Flip indices: {config.FLIP_INDICES}')

✓ All modules import cleanly
  Image size: 256
  Landmarks per face: 5
  Flip indices: (1, 0, 2, 4, 3)


In [6]:
# === Verify dataset files are in the workspace ===
import os
data_dir = f'{runtime.workspace_dir}/data'
print(f'Files in {data_dir}:')
for f in sorted(os.listdir(data_dir)):
    size_mb = os.path.getsize(f'{data_dir}/{f}') / (1024 * 1024)
    print(f'  {f}  ({size_mb:.1f} MB)')

Files in /Users/willvaughn/Developer/cv-face-alignment/data:
  face_alignment_test_images.npz  (74.8 MB)
  face_alignment_training_images.npz  (372.1 MB)
  train_val_split.npz  (0.0 MB)


In [7]:
# === Validate the data files can be loaded successfully ===
import numpy as np

train_path = f'{runtime.workspace_dir}/data/face_alignment_training_images.npz'
test_path = f'{runtime.workspace_dir}/data/face_alignment_test_images.npz'

# Load the training data using np.load
data_train = np.load(train_path, allow_pickle=True)
# Extract the images
images_train = data_train['images']
# and the data points
pts_train = data_train['points']
print(images_train.shape, pts_train.shape)

# same for test data (without labels)
data_test = np.load(test_path, allow_pickle=True)
images_test = data_test['images']
print(images_test.shape)
print('\n')



# Inspect the shape of the data
with np.load(train_path, allow_pickle=True) as data:
    print('Keys in the .npz:', list(data.keys()))
    for key in data.keys():
        arr = data[key]
        print(f'  {key}: shape={arr.shape}, dtype={arr.dtype}')


with np.load(test_path, allow_pickle=True) as data:
    print('\nTest keys:', list(data.keys()))
    for key in data.keys():
        arr = data[key]
        print(f'  {key}: shape={arr.shape}, dtype={arr.dtype}')

(2811, 256, 256, 3) (2811, 5, 2)
(554, 256, 256, 3)


Keys in the .npz: ['images', 'points']
  images: shape=(2811, 256, 256, 3), dtype=uint8
  points: shape=(2811, 5, 2), dtype=float64

Test keys: ['images']
  images: shape=(554, 256, 256, 3), dtype=uint8
